# Public review copy — MIS581 Capstone

This notebook is a sanitized publication copy of an executed development notebook. Local Windows paths were replaced with relative placeholders and embedded Census API configuration was changed to use the `CENSUS_API_KEY` environment variable. Historical error outputs were omitted; successful saved outputs were retained. The notebook has **not** been independently re-executed as a clean installation. It may require original source data, a compatible Python environment, and (for Notebook 08) PostgreSQL. Use the project README to understand dependencies and expected folder layout.


In [1]:
import os
import pandas as pd
import requests

# Census ACS year
year = 2024

# Census API key

api_key = os.getenv("CENSUS_API_KEY", "")
# Census ACS 5-Year API endpoint
base_url = f"https://api.census.gov/data/{year}/acs/acs5"

# Variables and geography requested from Census
params = {
    "get": "NAME,B01003_001E,B19013_001E",
    "for": "metropolitan statistical area/micropolitan statistical area:*",
    "key": api_key
}

# Send request to Census API
response = requests.get(
    base_url,
    params=params,
    timeout=30
)

# Check whether the request was successful
print("Status Code:", response.status_code)
print("Content Type:", response.headers.get("Content-Type"))

# Preview the beginning of the response
print(response.text[:300])

Status Code: 200
Content Type: application/json;charset=utf-8
[["NAME","B01003_001E","B19013_001E","metropolitan statistical area/micropolitan statistical area"],
["Aberdeen, SD Micro Area","41925","71488","10100"],
["Aberdeen, WA Micro Area","77053","64414","10140"],
["Abilene, TX Metro Area","180408","67328","10180"],
["Ada, OK Micro Area","38235","63017","1


In [2]:
data = response.json()

df_census = pd.DataFrame(
    data[1:],
    columns=data[0]
)

print(df_census.head())
print(df_census.shape)
print(df_census.dtypes)
df_census.info()

                      NAME B01003_001E B19013_001E  \
0  Aberdeen, SD Micro Area       41925       71488   
1  Aberdeen, WA Micro Area       77053       64414   
2   Abilene, TX Metro Area      180408       67328   
3       Ada, OK Micro Area       38235       63017   
4    Adrian, MI Micro Area       98415       70518   

  metropolitan statistical area/micropolitan statistical area  
0                                              10100           
1                                              10140           
2                                              10180           
3                                              10220           
4                                              10300           
(935, 4)
NAME                                                           object
B01003_001E                                                    object
B19013_001E                                                    object
metropolitan statistical area/micropolitan statistical area    object
dt

In [3]:
df_census = df_census.rename(columns={
    "NAME": "metro",
    "B01003_001E": "population",
    "B19013_001E": "median_household_income",
    "metropolitan statistical area/micropolitan statistical area": "cbsa"
})

print(df_census.head())
print(df_census.columns)

                     metro population median_household_income   cbsa
0  Aberdeen, SD Micro Area      41925                   71488  10100
1  Aberdeen, WA Micro Area      77053                   64414  10140
2   Abilene, TX Metro Area     180408                   67328  10180
3       Ada, OK Micro Area      38235                   63017  10220
4    Adrian, MI Micro Area      98415                   70518  10300
Index(['metro', 'population', 'median_household_income', 'cbsa'], dtype='object')


In [4]:
charleston_census = df_census[
    df_census["cbsa"] == "16700"
].copy()

print(charleston_census)

                                          metro population  \
157  Charleston-North Charleston, SC Metro Area     834267   

    median_household_income   cbsa  
157                   85254  16700  


In [5]:
census_metro = df_census[
    df_census["metro"].str.contains("Metro Area", na=False)
].copy()

print("\nCensus Metropolitan Areas Only")
print(census_metro.head())

print("\nNumber of Metro Areas:")
print(len(census_metro))

print("\nNumber of Micro Areas removed:")
print(len(df_census) - len(census_metro))


Census Metropolitan Areas Only
                       metro population median_household_income   cbsa
2     Abilene, TX Metro Area     180408                   67328  10180
5   Aguadilla, PR Metro Area     251769                   22398  10380
6       Akron, OH Metro Area     699820                   72371  10420
9      Albany, GA Metro Area     146018                   56906  10500
10     Albany, OR Metro Area     130706                   76329  10540

Number of Metro Areas:
393

Number of Micro Areas removed:
542


In [6]:
census_metro["population"] = pd.to_numeric(
    census_metro["population"],
    errors="coerce"
)

census_metro["median_household_income"] = pd.to_numeric(
    census_metro["median_household_income"],
    errors="coerce"
)

census_metro["cbsa"] = pd.to_numeric(
    census_metro["cbsa"],
    errors="coerce"
)

print(census_metro.dtypes)

print("\nMissing Values")
print(
    census_metro[
        ["population", "median_household_income", "cbsa"]
    ].isna().sum()
)

metro                      object
population                  int64
median_household_income     int64
cbsa                        int64
dtype: object

Missing Values
population                 0
median_household_income    0
cbsa                       0
dtype: int64


In [7]:
print("\nPopulation Statistics")
print(census_metro["population"].describe())

print("\nMedian Household Income Statistics")
print(census_metro["median_household_income"].describe())

print("\nLowest Median Household Income Values")
print(
    census_metro[
        ["metro", "cbsa", "median_household_income"]
    ]
    .sort_values("median_household_income")
    .head(10)
)


Population Statistics
count    3.930000e+02
mean     7.429407e+05
std      1.649234e+06
min      5.808200e+04
25%      1.463860e+05
50%      2.469890e+05
75%      5.777260e+05
max      1.979854e+07
Name: population, dtype: float64

Median Household Income Statistics
count       393.000000
mean      74545.577608
std       15733.238661
min       21163.000000
25%       65158.000000
50%       72475.000000
75%       81861.000000
max      162111.000000
Name: median_household_income, dtype: float64

Lowest Median Household Income Values
                                      metro   cbsa  median_household_income
530                 Mayagüez, PR Metro Area  32420                    21163
351                  Guayama, PR Metro Area  25020                    21186
677                    Ponce, PR Metro Area  38660                    22247
5                  Aguadilla, PR Metro Area  10380                    22398
39                   Arecibo, PR Metro Area  11640                    25084
755  Sa